# L4 — Orderbook Dynamics

Charts O1–O5.

**Memory strategy**:
- BBO: level==0 predicate pushdown
- O1/O4: minimal 3-column load per exchange, 2-hour window
- O3: 2-column load (exchange+received_at) — only counts needed
- O5: Polars lazy scan

In [ ]:
import os, gc, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_orderbook, load_orderbook_updates
from mda.layer4.bbo          import compute_bbo, spread_stats
from mda.layer4.update_rates import compute_update_rate_by_depth, compute_delta_compression_ratio
from mda.layer4.levels        import compute_level_lifetimes_polars, level_lifetime_stats
from mda.plots.charts import (
    plot_update_rate_by_depth, plot_bbo_spread_depth,
    plot_delta_compression_ts, plot_book_activity_heatmap,
    plot_level_lifetime_hist, save_figure,
)
print("imports OK")

In [ ]:
# Determine first 2-hour window from available parquet files
import os, re
ob_files = sorted(f for f in os.listdir(os.path.join(DATA_DIR, "orderbook")) if f.endswith(".parquet"))
first_ts = re.search(r'(\d{8}_\d{6})', ob_files[0]).group(1)
from datetime import datetime, timedelta, timezone
t0 = datetime.strptime(first_ts, "%Y%m%d_%H%M%S").replace(tzinfo=timezone.utc)
t2 = t0 + timedelta(hours=2)
START_DT = t0.strftime("%Y-%m-%dT%H:%M:%SZ")
END_DT   = t2.strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Analysis window: {START_DT} → {END_DT}")

## O2 — BBO Spread & Depth (level==0 only)

In [ ]:
ob_bbo = load_orderbook(DATA_DIR, bbo_only=True, start_dt=START_DT, end_dt=END_DT)
EXCHANGES = ob_bbo["exchange"].unique().tolist()
print(f"BBO rows: {len(ob_bbo):,} | exchanges: {EXCHANGES}")

In [ ]:
bbo = compute_bbo(ob_bbo)
s   = spread_stats(bbo)
print("BBO spread stats (bps):")
print(s.to_string(index=False))

In [ ]:
bbo["ts_dt"] = pd.to_datetime(bbo["received_at"], utc=True, format="mixed")
fig = plot_bbo_spread_depth(bbo)
fig.show()
save_figure(fig, "O2_bbo_spread_depth", REPORTS_DIR)
del ob_bbo, bbo, s
gc.collect()
print("BBO freed")

## O1/O4 — Update Rate by Depth (3-column load, 2-hour window)

In [ ]:
update_rate_parts = []
for exch in EXCHANGES:
    print(f"  {exch}...")
    ob_exch = load_orderbook(DATA_DIR, exchanges=[exch],
                             columns=["exchange", "level", "received_at"],
                             add_ts_cols=False,
                             start_dt=START_DT, end_dt=END_DT)
    # add_ts_cols=False: manually add receive_ts_dt for _resample_count_by_exchange
    ob_exch["receive_ts_dt"] = pd.to_datetime(ob_exch["received_at"], utc=True, format="mixed")
    print(f"    {len(ob_exch):,} rows")
    update_rate_parts.append(compute_update_rate_by_depth(ob_exch))
    del ob_exch
    gc.collect()

update_rate = pd.concat(update_rate_parts, ignore_index=True)
del update_rate_parts; gc.collect()
print("Update rate computed")
print(update_rate[update_rate["level"] < 10].to_string(index=False))

In [ ]:
fig = plot_update_rate_by_depth(update_rate[update_rate["level"] < 20])
fig.show()
save_figure(fig, "O1_update_rate_by_depth", REPORTS_DIR)

In [ ]:
for exch in EXCHANGES:
    fig = plot_book_activity_heatmap(update_rate, exchange=exch)
    fig.show()
    save_figure(fig, f"O4_book_activity_{exch}", REPORTS_DIR)
del update_rate; gc.collect()
print("update_rate freed")

## O3 — Delta Compression (counts only: 2-column load)

In [ ]:
# Only need row counts per time bin → load exchange + received_at only
ob_updates = load_orderbook_updates(DATA_DIR, start_dt=START_DT, end_dt=END_DT)
print(f"OB updates: {len(ob_updates):,} rows")

delta_parts = []
for exch in EXCHANGES:
    print(f"  {exch}...")
    # 2 columns only — _resample_count_by_exchange falls back to parsing received_at
    ob_exch = load_orderbook(DATA_DIR, exchanges=[exch],
                             columns=["exchange", "received_at"],
                             add_ts_cols=False,
                             start_dt=START_DT, end_dt=END_DT)
    ob_upd_exch = ob_updates[ob_updates["exchange"] == exch]
    try:
        delta_parts.append(compute_delta_compression_ratio(ob_exch, ob_upd_exch))
    except Exception as e:
        print(f"    skipped: {e}")
    del ob_exch; gc.collect()

del ob_updates; gc.collect()

if delta_parts:
    delta = pd.concat(delta_parts, ignore_index=True)
    del delta_parts
    fig = plot_delta_compression_ts(delta)
    fig.show()
    save_figure(fig, "O3_delta_compression_ts", REPORTS_DIR)

## O5 — Level Lifetimes (Polars lazy, 2-hour window)

In [ ]:
lifetime_parts = []
for exch in EXCHANGES:
    try:
        lt = compute_level_lifetimes_polars(DATA_DIR, exchange=exch,
                                            symbol="BTCUSDT")
        if lt is not None and len(lt) > 0:
            lifetime_parts.append(lt)
            print(f"{exch}: {len(lt):,} levels")
    except Exception as e:
        print(f"{exch}: skipped ({e})")

if lifetime_parts:
    lifetimes = pd.concat(lifetime_parts, ignore_index=True)
    print(level_lifetime_stats(lifetimes).to_string(index=False))
    fig = plot_level_lifetime_hist(lifetimes)
    fig.show()
    save_figure(fig, "O5_level_lifetime_hist", REPORTS_DIR)
else:
    print("No level lifetime data")